# Grilla focal de Madre de Dios

Se seleccionaron los 6 focos del clúster Suroeste. Su extensión es de aproximadamente 6.54 km de Este a Oeste y 5.33 km de Norte a Sur, por eso no entran en una ventana de 5 × 5 km. Se usa una ventana de 10 × 10 km con celdas de 100 m (100 × 100).

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import importlib.util

repo = Path('/content/wildfire-cellular-automata')
csv_firms = repo / 'data/raw/firms/firms_caso_estudio_seleccionado.csv'

if not csv_firms.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1', '--branch', 'main',
        'https://github.com/anthony-yt/wildfire-cellular-automata.git', str(repo)
    ], check=True)

if importlib.util.find_spec('rasterio') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'rasterio'])

os.chdir(repo)
print('Proyecto listo')

In [ ]:
import json
import numpy as np
import pandas as pd
import rasterio
from rasterio.transform import from_origin
from rasterio.warp import reproject, Resampling, transform as transformar
from rasterio.features import rasterize

archivos = {
    'dem': 'data/raw/dem/madre_de_dios_dem_50m.tif',
    'pendiente': 'data/raw/dem/madre_de_dios_slope_deg_50m.tif',
    'vegetacion': 'data/raw/vegetation/madre_de_dios_vegetacion_2024_50m.tif'
}

firms = pd.read_csv('data/raw/firms/firms_caso_estudio_seleccionado.csv')
ids = [
    'FIRMS-MDD-2024-0001', 'FIRMS-MDD-2024-0014',
    'FIRMS-MDD-2024-0016', 'FIRMS-MDD-2024-0017',
    'FIRMS-MDD-2024-0018', 'FIRMS-MDD-2024-0019'
]
focal = firms[firms['fire_id'].isin(ids)].copy()

assert len(firms) == 22
assert len(focal) == 6

with rasterio.open(archivos['dem']) as base:
    crs = base.crs
    limites = base.bounds
    forma_original = base.shape
    transform_original = base.transform

    for nombre, ruta in archivos.items():
        with rasterio.open(ruta) as src:
            assert src.crs == crs
            assert src.shape == forma_original
            assert src.transform == transform_original

print('DEM, pendiente y vegetación:', forma_original)
print('Resolución original: 50 m | CRS:', crs)
print('Focos FIRMS:', len(firms), '| Focos seleccionados:', len(focal))

In [ ]:
xs, ys = transformar(
    'EPSG:4326', crs,
    focal['longitude'].tolist(),
    focal['latitude'].tolist()
)

n = 100
res = 100
izquierda = (min(xs) + max(xs)) / 2 - 5000
superior = (min(ys) + max(ys)) / 2 + 5000
derecha = izquierda + 10000
inferior = superior - 10000
grid_transform = from_origin(izquierda, superior, res, res)

assert izquierda >= limites.left and derecha <= limites.right
assert inferior >= limites.bottom and superior <= limites.top

focal['x_utm_m'] = xs
focal['y_utm_m'] = ys
focal['grid_col'] = np.floor((np.array(xs) - izquierda) / res).astype(int)
focal['grid_row'] = np.floor((superior - np.array(ys)) / res).astype(int)
assert focal['grid_col'].between(0, 99).all()
assert focal['grid_row'].between(0, 99).all()

salida = Path('data/processed')
salida.mkdir(parents=True, exist_ok=True)

for nombre, ruta in archivos.items():
    with rasterio.open(ruta) as src:
        if nombre == 'vegetacion':
            destino = np.full((n, n), 255, dtype=np.uint8)
            nodata_src = None if src.nodata == 0 else src.nodata
            nodata_dst = 255
            metodo = Resampling.nearest
        else:
            destino = np.full((n, n), -9999, dtype=np.float32)
            nodata_src = src.nodata
            nodata_dst = -9999
            metodo = Resampling.bilinear

        reproject(
            source=src.read(1), destination=destino,
            src_transform=src.transform, src_crs=src.crs,
            src_nodata=nodata_src,
            dst_transform=grid_transform, dst_crs=crs,
            dst_nodata=nodata_dst, resampling=metodo
        )

        perfil = src.profile.copy()
        perfil.update(
            driver='GTiff', height=n, width=n, count=1,
            dtype=destino.dtype, crs=crs,
            transform=grid_transform, nodata=nodata_dst,
            compress='lzw'
        )
        with rasterio.open(salida / f'madre_de_dios_{nombre}_100m.tif', 'w', **perfil) as dst:
            dst.write(destino, 1)

formas = [
    ({'type': 'Point', 'coordinates': (float(x), float(y))}, 1)
    for x, y in zip(xs, ys)
]
mascara_firms = rasterize(
    formas, out_shape=(n, n), transform=grid_transform,
    fill=0, dtype='uint8'
)

with rasterio.open(
    salida / 'madre_de_dios_firms_100m.tif', 'w',
    driver='GTiff', height=n, width=n, count=1,
    dtype='uint8', crs=crs, transform=grid_transform,
    compress='lzw'
) as dst:
    dst.write(mascara_firms, 1)

focal.to_csv(salida / 'madre_de_dios_firms_100m.csv', index=False)

metadata = {
    'zona': 'Madre de Dios',
    'periodo': '2024-08-14/2024-08-18',
    'crs': str(crs),
    'shape': [n, n],
    'tamano_celda_m': res,
    'bbox_utm_m': {
        'xmin': float(izquierda), 'ymin': float(inferior),
        'xmax': float(derecha), 'ymax': float(superior)
    },
    'cantidad_focos': len(focal),
    'fire_ids': ids,
    'resolucion_original_m': 50,
    'fuente': 'NASA FIRMS VIIRS_SNPP_SP',
    'nota': 'Los focos FIRMS son detecciones de calor, no el perímetro real del área quemada.'
}
with open(salida / 'madre_de_dios_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=4, ensure_ascii=False)

print('Archivos generados en data/processed')

In [ ]:
nombres = ['dem', 'pendiente', 'vegetacion', 'firms']
transformaciones = []

for nombre in nombres:
    ruta = salida / f'madre_de_dios_{nombre}_100m.tif'
    with rasterio.open(ruta) as src:
        assert src.shape == (100, 100)
        assert src.crs == crs
        assert src.res == (100.0, 100.0)
        transformaciones.append(src.transform)

assert all(t == transformaciones[0] for t in transformaciones)
assert int(mascara_firms.sum()) == 6

print('Grilla: 100 x 100')
print('Resolución: 100 m')
print('Focos dentro: 6/6')
print('Capas alineadas: 4')
print('\nArchivos:')
for archivo in sorted(salida.glob('madre_de_dios_*')):
    print(archivo.name)